In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from src.custom_fastkan import FastKAN
import pandas as pd

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
def true_function(x):
    # f(x, y) = xy
    return x[:, 0] * x[:, 1]

torch.manual_seed(42)
num_samples = 10000

X = torch.rand(num_samples, 2) * 2 - 1 
y = true_function(X)

train_size = int(0.7 * num_samples)
val_size = int(0.15 * num_samples)
test_size = num_samples - train_size - val_size

X_train, X_val, X_test = torch.split(X, [train_size, val_size, test_size])
y_train, y_val, y_test = torch.split(y, [train_size, val_size, test_size])

X_train = X_train.to(device)
y_train = y_train.to(device)
X_val = X_val.to(device)
y_val = y_val.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Train shape: torch.Size([7000, 2]), Val shape: torch.Size([1500, 2]), Test shape: torch.Size([1500, 2])


In [3]:
model = FastKAN([2, 2, 1], grid_min=-1, grid_max=1, num_grids=10, use_base_update=False, use_layernorm=False).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [4]:
print("Training KAN...")
train_losses = []
val_losses = []

for epoch in range(1000):
    optimizer.zero_grad()
    pred = model(X_train).squeeze()
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        with torch.no_grad():
            val_pred = model(X_val).squeeze()
            val_loss = torch.mean((val_pred - y_val)**2)
            train_losses.append(loss.item())
            val_losses.append(val_loss.item())
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

model.eval()
with torch.no_grad():
    test_pred = model(X_test).squeeze()
    mse = mean_squared_error(y_test.cpu(), test_pred.cpu())
    r2 = r2_score(y_test.cpu(), test_pred.cpu())
    
    print(f"KAN Test MSE: {mse:.6f}")
    print(f"KAN Test R2: {r2:.4f}")

Training KAN...
Epoch 0, Train MSE: 0.114815, Val MSE: 0.116909
Epoch 50, Train MSE: 0.004716, Val MSE: 0.004333
Epoch 100, Train MSE: 0.000534, Val MSE: 0.000524
Epoch 150, Train MSE: 0.000183, Val MSE: 0.000190
Epoch 200, Train MSE: 0.000133, Val MSE: 0.000135
Epoch 250, Train MSE: 0.000125, Val MSE: 0.000127
Epoch 300, Train MSE: 0.000121, Val MSE: 0.000123
Epoch 350, Train MSE: 0.000119, Val MSE: 0.000121
Epoch 400, Train MSE: 0.000117, Val MSE: 0.000120
Epoch 450, Train MSE: 0.000117, Val MSE: 0.000120
Epoch 500, Train MSE: 0.000116, Val MSE: 0.000119
Epoch 550, Train MSE: 0.000115, Val MSE: 0.000119
Epoch 600, Train MSE: 0.000115, Val MSE: 0.000119
Epoch 650, Train MSE: 0.000115, Val MSE: 0.000118
Epoch 700, Train MSE: 0.000114, Val MSE: 0.000118
Epoch 750, Train MSE: 0.000114, Val MSE: 0.000118
Epoch 800, Train MSE: 0.000114, Val MSE: 0.000118
Epoch 850, Train MSE: 0.000114, Val MSE: 0.000118
Epoch 900, Train MSE: 0.000114, Val MSE: 0.000118
Epoch 950, Train MSE: 0.000113, Val M

In [5]:
import torch.nn as nn
torch.manual_seed(42)
class MLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dims=[64, 64], output_dim=1):
        super(MLP, self).__init__()
        layers = []
        curr_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(curr_dim, h_dim))
            layers.append(nn.ReLU())
            curr_dim = h_dim
        layers.append(nn.Linear(curr_dim, output_dim))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

mlp_model = MLP(input_dim=2, hidden_dims=[32], output_dim=1).to(device)
mlp_optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.01)

print("Training MLP...")
for epoch in range(1000):
    mlp_model.train()
    mlp_optimizer.zero_grad()
    pred = mlp_model(X_train).squeeze()
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    mlp_optimizer.step()
    
    if epoch % 100 == 0:
        mlp_model.eval()
        with torch.no_grad():
            val_pred = mlp_model(X_val).squeeze()
            val_loss = torch.mean((val_pred - y_val)**2)
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

mlp_model.eval()
with torch.no_grad():
    mlp_pred = mlp_model(X_test).squeeze()
    mlp_mse = mean_squared_error(y_test.cpu(), mlp_pred.cpu())
    mlp_r2 = r2_score(y_test.cpu(), mlp_pred.cpu())
    
    print(f"\nMLP Test MSE: {mlp_mse:.6f}")
    print(f"MLP Test R2: {mlp_r2:.4f}")

Training MLP...
Epoch 0, Train MSE: 0.109862, Val MSE: 0.102521
Epoch 100, Train MSE: 0.001269, Val MSE: 0.001314
Epoch 200, Train MSE: 0.000543, Val MSE: 0.000562
Epoch 300, Train MSE: 0.000307, Val MSE: 0.000304
Epoch 400, Train MSE: 0.000208, Val MSE: 0.000203
Epoch 500, Train MSE: 0.000157, Val MSE: 0.000153
Epoch 600, Train MSE: 0.000130, Val MSE: 0.000131
Epoch 700, Train MSE: 0.000186, Val MSE: 0.000244
Epoch 800, Train MSE: 0.000103, Val MSE: 0.000104
Epoch 900, Train MSE: 0.000195, Val MSE: 0.000127

MLP Test MSE: 0.000089
MLP Test R2: 0.9992


In [6]:
print("\nAnalyzing individual KAN prediction losses...")
individual_losses = []
predictions = []
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        individual_losses.append(loss)
        predictions.append(prediction)

individual_losses = np.array(individual_losses)
predictions = torch.stack(predictions)
sorted_indices = np.argsort(individual_losses)

mean_loss = np.mean(individual_losses)
lowest_indices = sorted_indices[:3] 
highest_indices = sorted_indices[-3:]

mean_distances = np.abs(individual_losses - mean_loss)
mean_sorted_indices = np.argsort(mean_distances)
mean_indices = mean_sorted_indices[:3]

print(f"\nKAN Loss Statistics:")
print(f"Mean Loss: {mean_loss:.6f}")
print(f"Min Loss: {individual_losses[lowest_indices[0]]:.6f}")
print(f"Max Loss: {individual_losses[highest_indices[-1]]:.6f}")


Analyzing individual KAN prediction losses...

KAN Loss Statistics:
Mean Loss: 0.000111
Min Loss: 0.000000
Max Loss: 0.006247


In [7]:
print("\nAnalyzing individual MLP prediction losses...")
mlp_individual_losses = []
mlp_predictions = []

mlp_model.eval()
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = mlp_model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        mlp_individual_losses.append(loss)
        mlp_predictions.append(prediction)

mlp_individual_losses = np.array(mlp_individual_losses)
mlp_predictions = torch.stack(mlp_predictions)
mlp_sorted_indices = np.argsort(mlp_individual_losses)

mlp_mean_loss = np.mean(mlp_individual_losses)
mlp_lowest_indices = mlp_sorted_indices[:3] 
mlp_highest_indices = mlp_sorted_indices[-3:]

mlp_mean_distances = np.abs(mlp_individual_losses - mlp_mean_loss)
mlp_mean_sorted_indices = np.argsort(mlp_mean_distances)
mlp_mean_indices = mlp_mean_sorted_indices[:3]

print(f"\nMLP Loss Statistics:")
print(f"Mean Loss: {mlp_mean_loss:.6f}")
print(f"Min Loss: {mlp_individual_losses[mlp_lowest_indices[0]]:.6f}")
print(f"Max Loss: {mlp_individual_losses[mlp_highest_indices[-1]]:.6f}")


Analyzing individual MLP prediction losses...

MLP Loss Statistics:
Mean Loss: 0.000089
Min Loss: 0.000000
Max Loss: 0.002879


In [8]:
table_data = []
categories = [("Lowest", lowest_indices), ("Highest", highest_indices), ("Mean", mean_indices)]

for label, indices in categories:
    for idx in indices:
        table_data.append({
            "Category": label,
            "Index": idx,
            "Input (x, y)": f"({X_test[idx][0].item():.4f}, {X_test[idx][1].item():.4f})",
            "True Value": y_test[idx].item(),
            "Predicted": predictions[idx].item(),
            "Loss": individual_losses[idx]
        })

df_kan_analysis = pd.DataFrame(table_data)
df_kan_analysis


,Category,Index,"Input (x, y)",True Value,Predicted,Loss
0,Lowest,150,"(-0.1597, 0.1990)",-0.031786,-0.031787,1.758815e-12
1,Lowest,1497,"(-0.2855, -0.0416)",0.011893,0.011885,5.765476e-11
2,Lowest,795,"(0.9498, -0.4807)",-0.456554,-0.456543,1.183276e-10
3,Highest,556,"(0.8747, -0.8838)",-0.773105,-0.830379,3.280378e-03
4,Highest,548,"(0.9998, 0.7501)",0.749974,0.687813,3.863919e-03
5,Highest,493,"(-0.9972, 0.9704)",-0.967742,-0.888706,6.246614e-03
6,Mean,822,"(0.5068, -0.5370)",-0.272152,-0.282685,1.109385e-04
7,Mean,100,"(-0.7981, -0.4903)",0.391299,0.401877,1.118873e-04
8,Mean,733,"(-0.5025, -0.4955)",0.249021,0.259537,1.105960e-04


In [9]:
table_data_mlp = []
categories = [("Lowest", mlp_lowest_indices), ("Highest", mlp_highest_indices), ("Mean", mlp_mean_indices)]

for label, indices in categories:
    for idx in indices:
        table_data_mlp.append({
            "Category": label,
            "Index": idx,
            "Input (x, y)": f"({X_test[idx][0].item():.4f}, {X_test[idx][1].item():.4f})",
            "True Value": y_test[idx].item(),
            "Predicted": mlp_predictions[idx].item(),
            "Loss": mlp_individual_losses[idx]
        })

df_mlp_analysis = pd.DataFrame(table_data_mlp)
df_mlp_analysis


,Category,Index,"Input (x, y)",True Value,Predicted,Loss
0,Lowest,1005,"(0.4020, 0.8760)",0.352148,0.352162,1.920464e-10
1,Lowest,1422,"(0.7297, 0.2001)",0.145979,0.145995,2.825260e-10
2,Lowest,1072,"(0.4909, 0.7000)",0.343635,0.343617,3.326619e-10
3,Highest,1307,"(-0.8917, -0.9996)",0.891354,0.845049,2.144181e-03
4,Highest,839,"(-0.9836, -0.9470)",0.931411,0.879861,2.657314e-03
5,Highest,843,"(-0.9877, -0.9504)",0.938765,0.885106,2.879297e-03
6,Mean,1366,"(0.0456, 0.5673)",0.025876,0.035320,8.918534e-05
7,Mean,1369,"(0.3870, 0.4453)",0.172340,0.162936,8.842632e-05
8,Mean,420,"(0.8603, 0.3973)",0.341795,0.351253,8.945669e-05


In [10]:
mlp_save_path = "model_pkls/functionxy_mlp_model.pkl"
torch.save({
    'model_state_dict': mlp_model.state_dict(),
    'config': {
        'input_dim': 2,
        'hidden_dims': [32],
        'output_dim': 1
    }
}, mlp_save_path)
print(f"MLP Model saved to {mlp_save_path}")

MLP Model saved to model_pkls/functionxy_mlp_model.pkl


In [11]:
kan_save_path = "model_pkls/functionxy_kan_model.pkl"
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'layers_hidden': [2, 2, 1],
        'grid_min': -1,
        'grid_max': 1,
        'num_grids': 10,
        'use_base_update': False,
        'use_layernorm': False,
    }
}, kan_save_path)
print(f"KAN Model saved to {kan_save_path}")

KAN Model saved to model_pkls/functionxy_kan_model.pkl
